<a href="https://colab.research.google.com/github/Khairul-islam99/Bangla-TTS/blob/master/Multimodal_LLM_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# Install required dependencies
!pip install datasets -q
!pip install gradio -q

import os, re

# Optimized installations for Google Colab
if "COLAB_" in "".join(os.environ.keys()):
    import torch
    v = re.match(r"[0-9\.]*", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

# Pin versions for stability
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2

**Dataset**

We will use the `gsm8k` dataset, which contains high-quality mathematical word problems. Since our model expects a conversational format, we will convert the raw questions and answers into a standard `user`/`assistant` chat format.

In [ ]:
import json
from datasets import load_dataset, Dataset

# Load the "main" subset of the gsm8k dataset
raw_train_dataset = load_dataset("gsm8k", "main", split="train")

def convert_math_conversation(sample):
    """Converts the gsm8k QA pairs into standard conversational JSON arrays."""
    conversation = [
        {
            "role": "user",
            "content": [{"type": "text", "text": sample["question"]}]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}]
        },
    ]
    return { "messages": conversation }

# Convert and save the training dataset
converted_math_dataset = [convert_math_conversation(sample) for sample in raw_train_dataset]
jsonl_filename = "converted_math_dataset.jsonl"

with open(jsonl_filename, 'w') as f:
    for item in converted_math_dataset:
        f.write(json.dumps(item) + '\n')

print(f"✅ Successfully converted and saved {len(converted_math_dataset)} training samples to '{jsonl_filename}'.")

# Load it back as a Hugging Face Dataset object
train_dataset = load_dataset('json', data_files=jsonl_filename, split='train')

# Prepare Evaluation Dataset
eval_dataset_raw = load_dataset("gsm8k", "main", split="test")
converted_eval_dataset = [convert_math_conversation(sample) for sample in eval_dataset_raw]
eval_dataset = Dataset.from_list(converted_eval_dataset)

print(f"✅ Evaluation dataset ready: {len(eval_dataset)} samples.")

In [ ]:
from unsloth import FastVisionModel
import torch

# Load Base Multimodal Model
model, tokenizer = FastVisionModel.from_pretrained(
    "khairul5/BINI-Qwen3-VL-MyMirror",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Apply PEFT / LoRA Adapters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,           # LoRA Rank
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

print("\n✅ Model and LoRA adapters successfully loaded.")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

# Setup Data Collator for Qwen3-VL
data_collator = UnslothVisionDataCollator(model, tokenizer)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    data_collator=data_collator,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        optim="adamw_8bit",
        logging_steps=10,
        output_dir="outputs",
        warmup_steps=5,
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none", # Set to "wandb" if you wish to track metrics

        # Vision Model Specifics
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)

print("🚀 Starting fine-tuning...")
trainer_stats = trainer.train()
print("✅ Fine-tuning complete!")

# Display Memory Statistics
gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)

print(f"GPU: {gpu_stats.name} | Max Memory: {max_memory} GB")
print(f"Peak Reserved Memory during training: {used_memory} GB")

In [ ]:
import shutil
from google.colab import files

print("📊 Running evaluation on test set...")
eval_results = trainer.evaluate(eval_dataset)
print(f"\n--- Evaluation Results ---")
print(f"Eval Loss: {eval_results['eval_loss']:.4f}")

# Save Adapters
output_dir = "bini_math_model_adapters"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Adapters saved to {output_dir}. Zipping files...")

# Zip and Download
shutil.make_archive(output_dir, 'zip', output_dir)
files.download(f"{output_dir}.zip")

Interactive Inference (Gradio)

In [ ]:
import gradio as gr
from threading import Thread
from transformers import TextIteratorStreamer

# Set model to inference mode
FastVisionModel.for_inference(model)
print("✅ Model is ready for inference.")

def format_chat_history(chat_history):
    messages = []
    if not chat_history: return messages
    for user_msg, assistant_msg in chat_history:
        messages.append({"role": "user", "content": [{"type": "text", "text": user_msg}]})
        if assistant_msg:
             messages.append({"role": "assistant", "content": [{"type": "text", "text": assistant_msg}]})
    return messages

def chat_stream(image_input, text_input, chat_history):
    messages = format_chat_history(chat_history)
    new_user_content = []

    if image_input:
        new_user_content.append({"type": "image"})

    if text_input:
        new_user_content.append({"type": "text", "text": text_input})
    elif image_input:
        text_input = "Describe this image."
        new_user_content.append({"type": "text", "text": text_input})
    else:
        yield chat_history
        return

    messages.append({"role": "user", "content": new_user_content})
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        image_input,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=1024,
        use_cache=True,
        temperature=0.7,
        min_p=0.1
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    if chat_history is None: chat_history = []
    chat_history.append((text_input, ""))

    response_text = ""
    for new_token in streamer:
        response_text += new_token
        chat_history[-1] = (text_input, response_text)
        yield chat_history

def clear_chat():
    return [], None, ""

# --- Gradio UI Layout ---
theme = gr.themes.Soft(primary_hue="blue", secondary_hue="indigo")

with gr.Blocks(theme=theme, title="Bini Math Assistant") as demo:
    gr.Markdown(
        """
        <div style="text-align: center;">
            <h1>🤖 Bini (Math Fine-Tuned)</h1>
            <p>Your personal multimodal assistant, optimized for mathematical reasoning!</p>
        </div>
        """
    )

    chatbot = gr.Chatbot(
        value=[[None, "Hello! I'm Bini. I've been fine-tuned on math problems. How can I help?"]],
        label="Chat",
        height=500
    )

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=200):
            image_box = gr.Image(type="pil", label="Upload Image (Optional)", sources=["upload"], height=150)
        with gr.Column(scale=4):
            text_box = gr.Textbox(label="Send a message", placeholder="Type your math problem...", show_label=False, lines=3)

    with gr.Row():
        clear_btn = gr.Button("Clear Chat", variant="secondary")
        send_btn = gr.Button("Send", variant="primary")

    gr.Examples(
        examples=[
            ["Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"],
            ["What is the derivative of x^2 + 2x?"],
        ],
        inputs=[text_box],
        label="Example Prompts"
    )

    gr.Markdown(
        """
        <div style="text-align: center; font-size: small; color: grey;">
            <p><b>Bini</b> was built by <b>Md Khairrul Islam</b>.<br>
            Powered by Unsloth, Qwen3-VL-8B-Instruct, and Gradio.</p>
        </div>
        """
    )

    # Event Handlers
    send_btn.click(chat_stream, inputs=[image_box, text_box, chatbot], outputs=[chatbot]) \
            .then(lambda: (None, ""), outputs=[image_box, text_box])

    text_box.submit(chat_stream, inputs=[image_box, text_box, chatbot], outputs=[chatbot]) \
            .then(lambda: (None, ""), outputs=[image_box, text_box])

    clear_btn.click(clear_chat, outputs=[chatbot, image_box, text_box])

demo.launch(debug=True, share=True)